In [2]:
import numpy as np
import pandas as pd

# 1. Load the merged dataset
df = pd.read_csv("DOAS_urdaneta_2014_2021_combined.csv")

# 2. Standardize column names (trim whitespace and convert to lowercase)
df.columns = df.columns.str.strip().str.lower()

# 3. Parse and standardize dates, then sort chronologically
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(by="date").reset_index(drop=True)

# 4. Convert pollutant columns to numeric and handle invalid string symbols
# (e.g., '-', '_', non-breaking dashes '\x96', Excel errors '#DIV/0!')
pollutant_cols = ["so2", "nox", "o3", "pm2.5", "pm10", "co"]

for col in pollutant_cols:
  df[col] = pd.to_numeric(df[col], errors="coerce")

  # Environmental concentrations cannot be negative; replace sensor errors/flags (-999, negative readings) with NaN
  df.loc[df[col] < 0, col] = np.nan

# 5. Handle missing values (optional: linear interpolation for time-series continuity)
# Limit consecutive interpolation to avoid filling long sensor down periods
df[pollutant_cols] = df[pollutant_cols].interpolate(
    method="linear", limit=7, limit_direction="both"
)

# 6. Save the cleaned dataset
df.to_csv("cleaned_DOAS_urdaneta_2014_2021.csv", index=False)